# Myeloid Cell Annotation Validation with Enhanced Marker Panel

**Purpose**: Validate existing LLM-based myeloid annotations using expanded discriminative marker sets and signature scoring

**Key Improvements**:
- Discriminative markers for fine-grained myeloid subtypes (AM vs IM vs moMac, cDC1 vs cDC2 vs DC3 vs migDC)
- Signature scoring approach (more robust than single genes)
- State-based markers (IFN, inflammation, hypoxia, proliferation)
- Comprehensive contamination panel including RBC, platelets, plasma cells
- Cluster-level signature heatmaps for global patterns
- Per-cluster validation reports with confidence scoring

**Validation Strategy**:
1. Check if assigned cell types match expected marker signatures
2. Identify potential doublets (dual-lineage signature enrichment)
3. Detect contamination (cross-lineage marker expression)
4. Flag clusters with low confidence for manual review

**Output**:
- Signature score heatmaps (cluster × signature)
- Annotation confidence scores
- Validation reports per cluster
- Flagged clusters requiring review

## 1. Configuration & Setup

In [ ]:
# ===== Import Libraries =====
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from scipy import sparse
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

In [ ]:
# ===== Configuration =====
# Paths
INPUT_PATH = "/home/h2048/data/py/0128/myeloid_analysis_unified/results/subcluster_unified_v2_20260128/adata_myeloid_subclustered_FINAL_v2_20260128.h5ad"
OUTPUT_DIR = Path("/home/h2048/data/py/0209/myeloid_validation_enhanced")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Column names
CELLTYPE_COL = 'cell_type_scanvi_filt'  # Original annotation to validate
BATCH_KEY = 'sample'

# Scanpy settings
sc.settings.verbosity = 1
sc.settings.n_jobs = 48
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(8, 6))

print(f"Output directory: {OUTPUT_DIR}")
print(f"Configuration complete")

## 2. Enhanced Marker Gene Sets

### Discriminative Marker Strategy:
- **Pan-markers**: Too broad (e.g., LYZ in all myeloid)
- **Discriminative markers**: Distinguish between closely related subtypes
- **State markers**: Capture functional states across subtypes
- **Contamination markers**: Detect cross-lineage expression

In [ ]:
# ===== Core Myeloid Identity =====
CORE_MYELOID = {
    'Pan_Leukocyte': ['PTPRC'],  # CD45
    'Myeloid_Program': ['SPI1', 'IRF8', 'CSF1R', 'ITGAM', 'FCGR2A', 'CTSS'],
    'Pan_Myeloid_Basic': ['LYZ', 'LST1', 'TYROBP', 'FCER1G', 'AIF1'],
    'APC_Core': ['HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DPB1', 'CD74'],
    'Complement_Phagocytosis': ['C1QA', 'C1QB', 'C1QC', 'APOE'],
}

# ===== Discriminative Myeloid Subtypes =====
MYELOID_SUBTYPES_ENHANCED = {
    # Monocyte subtypes
    'Classical_Mono': ['S100A8', 'S100A9', 'S100A12', 'FCN1', 'VCAN', 'CD14', 'LGALS3'],
    'Intermediate_Mono': ['CD14', 'LYZ', 'LST1', 'FCGR3A', 'MS4A7', 'HLA-DRA'],
    'Non_Classical_Mono': ['FCGR3A', 'CX3CR1', 'LST1', 'MS4A7', 'IFITM3', 'LILRB1', 'LILRB2'],
    
    # Macrophage subtypes
    'Alveolar_Mac': ['MARCO', 'PPARG', 'FABP4', 'LPL', 'APOE', 'APOC1', 'MSR1', 'CSF2RA', 'ITGAX'],
    'Interstitial_Resident_Mac': ['LYVE1', 'FOLR2', 'MRC1', 'CD163', 'MERTK', 'SIGLEC1', 'SEPP1'],
    'MonoDerived_Inflammatory_Mac': ['SPP1', 'CHI3L1', 'FN1', 'VCAN', 'IL1B', 'CXCL8', 'MMP9'],
    'TREM2_Lipid_Mac': ['TREM2', 'APOE', 'LPL', 'GPNMB', 'LGALS3'],
    
    # DC subtypes
    'cDC1': ['XCR1', 'CLEC9A', 'BATF3', 'IRF8', 'CADM1'],
    'cDC2': ['CD1C', 'FCER1A', 'CLEC10A', 'CD1E', 'IRF4'],
    'DC3_MonoDC': ['CD1C', 'FCER1A', 'CD14', 'LST1', 'FCGR3A', 'S100A8'],
    'pDC': ['IL3RA', 'GZMB', 'TCF4', 'CLEC4C', 'IRF7', 'LILRA4'],
    'Migratory_LAMP3_DC': ['LAMP3', 'CCR7', 'FSCN1', 'RELB', 'CCL17', 'CCL22'],
    
    # Granulocytes
    'Neutrophil': ['CSF3R', 'FCGR3B', 'CXCR2', 'MPO', 'ELANE', 'AZU1', 'CXCR1'],
    'Eosinophil': ['SIGLEC8', 'IL5RA', 'PRG2', 'RNASE2', 'RNASE3', 'CLC'],
    'Mast': ['KIT', 'TPSAB1', 'TPSB2', 'CPA3', 'HDC', 'MS4A2'],
}

# ===== Functional State Signatures =====
MYELOID_STATES = {
    'MHCII_Antigen_Presentation': ['HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DPB1', 'CD74'],
    'Type_I_IFN_Response': ['ISG15', 'IFIT1', 'IFIT3', 'MX1', 'OAS1', 'OASL', 'RSAD2', 'IRF7'],
    'IL1_NFKB_Inflammation': ['IL1B', 'NLRP3', 'TNF', 'NFKBIA', 'CXCL2', 'CCL3', 'CCL4'],
    'Hypoxia_Glycolysis': ['SLC2A1', 'LDHA', 'HK2', 'PFKFB3'],
    'Proliferation': ['MKI67', 'TOP2A', 'STMN1', 'HMGB2', 'PCNA'],
    'Chemokine_Production': ['CCL2', 'CCL3', 'CCL4', 'CXCL2', 'CXCL8', 'CXCL10'],
}

# ===== Comprehensive Contamination Markers =====
CONTAMINATION_MARKERS_ENHANCED = {
    # Epithelial
    'Epithelial_General': ['EPCAM', 'KRT19', 'KRT18', 'KRT8', 'CDH1'],
    'Airway_Epithelial': ['BPIFA1', 'SCGB1A1', 'MUC5AC', 'FOXJ1'],
    'Alveolar_Epithelial': ['SFTPA1', 'SFTPA2', 'SFTPB', 'SFTPC', 'SFTPD'],
    
    # Lymphocytes
    'B_Cell': ['MS4A1', 'CD79A', 'CD79B', 'IGHM', 'IGKC', 'IGLC1'],
    'Plasma_Cell': ['MZB1', 'XBP1', 'JCHAIN', 'SDC1'],
    'T_Cell': ['CD3D', 'CD3E', 'CD4', 'CD8A', 'IL7R'],
    'NK_Cell': ['NKG7', 'GNLY', 'PRF1', 'GZMB', 'NCAM1'],
    
    # Stromal/Vascular
    'Endothelial': ['PECAM1', 'VWF', 'KDR', 'EMCN', 'CDH5'],
    'Lymphatic_Endothelial': ['PROX1', 'PDPN', 'LYVE1', 'CCL21'],
    'Fibroblast': ['COL1A1', 'COL3A1', 'DCN', 'LUM'],
    'Pericyte': ['RGS5', 'PDGFRB', 'ACTA2'],
    'Smooth_Muscle': ['ACTA2', 'TAGLN', 'MYH11'],
    
    # Technical contaminants
    'RBC': ['HBB', 'HBA1', 'HBA2', 'ALAS2'],
    'Platelet': ['PPBP', 'PF4', 'GP9'],
}

# ===== Problem-Specific Marker Panels =====
# For targeted validation of suspected issues
PROBLEM_SPECIFIC_PANELS = {
    'AM_vs_AM_Endo_doublet': {
        'AM_side': ['MARCO', 'PPARG', 'FABP4', 'LPL', 'APOE', 'CSF2RA'],
        'Endo_side': ['PECAM1', 'VWF', 'EMCN', 'KDR', 'CDH5'],
        'Lymphatic_Endo': ['PROX1', 'PDPN', 'CCL21'],
    },
    'DC_vs_DC_Epi_doublet': {
        'DC_core': ['HLA-DRA', 'CD74', 'FCER1A', 'CD1C'],
        'Migratory_DC': ['LAMP3', 'CCR7', 'FSCN1'],
        'Epi_side': ['EPCAM', 'KRT8', 'KRT18', 'KRT19', 'BPIFA1', 'SFTPC'],
    },
    'Mono_vs_Mono_B_doublet': {
        'Mono_side': ['FCGR3A', 'LST1', 'CX3CR1', 'MS4A7'],
        'B_Plasma_side': ['MS4A1', 'CD79A', 'IGKC', 'MZB1', 'JCHAIN'],
    },
    'True_Mac_vs_Contaminant': {
        'Core_Mac': ['C1QA', 'C1QB', 'C1QC', 'APOE', 'LST1', 'TYROBP'],
        'Cross_lineage': ['EPCAM', 'KRT19', 'PECAM1', 'VWF'],
    },
}

print("Enhanced marker gene sets defined:")
print(f"  Core myeloid categories: {len(CORE_MYELOID)}")
print(f"  Myeloid subtypes: {len(MYELOID_SUBTYPES_ENHANCED)}")
print(f"  Functional states: {len(MYELOID_STATES)}")
print(f"  Contamination categories: {len(CONTAMINATION_MARKERS_ENHANCED)}")
print(f"  Problem-specific panels: {len(PROBLEM_SPECIFIC_PANELS)}")

## 3. Load Data

In [ ]:
# ===== Load Dataset =====
print("Loading myeloid dataset...")
adata = sc.read_h5ad(INPUT_PATH)

print(f"Dataset loaded: {adata.n_obs:,} cells × {adata.n_vars} genes")
print(f"\nData structure:")
print(f"  Layers: {list(adata.layers.keys())}")
print(f"  Obsm: {list(adata.obsm.keys())}")
print(f"  .raw present: {adata.raw is not None}")
if adata.raw is not None:
    print(f"  .raw genes: {adata.raw.n_vars}")

In [ ]:
# =============================================================================
# Myeloid Cell Type L3 Label Standardization (Python / AnnData)
# =============================================================================
# Purpose: Convert numeric L3 subcluster IDs to hierarchical format
# Format:  {cell_type_L2}_c{subcluster_id}
# Example: "Alveolar macrophages_c0", "Classical monocytes_c1"
# =============================================================================

import numpy as np
import pandas as pd

def standardize_myeloid_l3_labels(
    adata,
    l2_key: str = "cell_type_L2",
    l3_key: str = "cell_type_L3",
    subcluster_key: str = "subcluster_id",
    backup_key: str = "cell_type_L3_original",
    hierarchical_key: str = "cell_type_L3_hierarchical",
    verbose: bool = True,
):
    """
    In-place standardization of L3 labels in AnnData:
      1) Preserve original L3
      2) Parse numeric subcluster_id from existing L3
      3) Create hierarchical L3 = {L2}_c{subcluster_id}
      4) Order L3 categorical levels by (L2 alphabetical, subcluster_id numeric)
      5) Print validation summaries

    Parameters
    ----------
    adata : AnnData
        Must have adata.obs[l2_key] and adata.obs[l3_key]
    """
    line = "=" * 80
    if verbose:
        print("\n" + line)
        print("MYELOID L3 LABEL STANDARDIZATION (Python / AnnData)")
        print(line + "\n")

    # -----------------------------
    # Step 1: Preserve Original L3
    # -----------------------------
    if verbose:
        print("Step 1: Preserving original subcluster IDs...\n")
        print("Original L3 labels (top 20):")

    if l3_key not in adata.obs.columns:
        raise KeyError(f"'{l3_key}' not found in adata.obs")

    original_counts = adata.obs[l3_key].value_counts(dropna=False)
    if verbose:
        print(original_counts.head(20))
        if len(original_counts) > 20:
            print(f"  ... and {len(original_counts) - 20} more")

    # Backup original L3 labels
    adata.obs[backup_key] = adata.obs[l3_key].astype("object")

    # Parse numeric subcluster IDs (R: as.integer(as.character(...)))
    # Use nullable Int64 to preserve NA cleanly
    adata.obs[subcluster_key] = pd.to_numeric(
        adata.obs[l3_key].astype("string"),
        errors="coerce"
    ).astype("Int64")

    n_unique_ids = adata.obs[subcluster_key].dropna().nunique()
    if verbose:
        print(f"\n✓ Preserved {n_unique_ids} unique subcluster IDs")

    # -----------------------------
    # Step 2: Create Hierarchical L3
    # -----------------------------
    if verbose:
        print("\nStep 2: Creating hierarchical L3 labels...")

    if l2_key not in adata.obs.columns:
        raise KeyError(f"'{l2_key}' not found in adata.obs")

    l2 = adata.obs[l2_key].astype("string")
    sid = adata.obs[subcluster_key]

    # hierarchical label: {L2}_c{subcluster_id}
    adata.obs[hierarchical_key] = pd.Series(pd.NA, index=adata.obs_names, dtype="string")
    mask = l2.notna() & sid.notna()
    adata.obs.loc[mask, hierarchical_key] = (l2[mask] + "_c" + sid[mask].astype("Int64").astype("string"))

    # Replace L3 with hierarchical format
    adata.obs[l3_key] = adata.obs[hierarchical_key].astype("object")

    if verbose:
        print("✓ Created hierarchical L3 labels")

    # -----------------------------
    # Step 3: Order Factor Levels
    # -----------------------------
    if verbose:
        print("\nStep 3: Ordering L3 categorical levels...")

    tmp_df = adata.obs[[l2_key, subcluster_key, l3_key]].copy()
    tmp_df = tmp_df.dropna(subset=[l3_key, l2_key, subcluster_key])

    # sort by L2 (alphabetical) then subcluster_id (numeric)
    tmp_df = tmp_df.sort_values(
        by=[l2_key, subcluster_key],
        key=lambda s: s.astype("string") if s.name == l2_key else s
    )

    ordered_levels = pd.unique(tmp_df[l3_key].astype("string")).tolist()

    # Apply ordered categorical
    adata.obs[l3_key] = pd.Categorical(
        adata.obs[l3_key].astype("string"),
        categories=ordered_levels,
        ordered=True
    )

    if verbose:
        print(f"✓ Ordered {len(ordered_levels)} L3 levels")

    # -----------------------------
    # Step 4: Validation
    # -----------------------------
    if verbose:
        print("\n" + line)
        print("VALIDATION RESULTS")
        print(line + "\n")

        total_cells = adata.n_obs
        valid_l3 = adata.obs[l3_key].notna().sum()

        print("Overall Statistics:")
        print(f"  Total cells: {total_cells:,}")
        print(f"  Cells with valid L3 labels: {valid_l3:,} ({valid_l3/total_cells*100:.1f}%)")

        # Number of unique subclusters = number of categorical levels (if categorical)
        if isinstance(adata.obs[l3_key].dtype, pd.CategoricalDtype):
            n_levels = len(adata.obs[l3_key].cat.categories)
        else:
            n_levels = adata.obs[l3_key].nunique(dropna=True)
        print(f"  Unique L3 subclusters: {n_levels}")

        print("\nFinal L3 Label Distribution:")
        print("-" * 80)
        final_counts = pd.Series(adata.obs[l3_key]).value_counts(dropna=False)
        print(final_counts)

    # -----------------------------
    # Step 5: Per-Celltype Summary
    # -----------------------------
    if verbose:
        print("\n" + line)
        print("PER-CELLTYPE SUBCLUSTER SUMMARY")
        print(line + "\n")

    df = adata.obs[[l2_key, l3_key, subcluster_key]].copy()
    df = df.dropna(subset=[l2_key, l3_key, subcluster_key])

    # Ensure subcluster_id is numeric for min/max
    df[subcluster_key] = df[subcluster_key].astype("Int64")

    l2_summary = (
        df.groupby(l2_key, dropna=False)
          .agg(
              n_cells=(l2_key, "size"),
              n_subclusters=(subcluster_key, pd.Series.nunique),
              min_id=(subcluster_key, "min"),
              max_id=(subcluster_key, "max"),
          )
          .reset_index()
    )
    l2_summary["subcluster_range"] = "c" + l2_summary["min_id"].astype("string") + "-c" + l2_summary["max_id"].astype("string")
    l2_summary["avg_cells_per_subcluster"] = (l2_summary["n_cells"] / l2_summary["n_subclusters"]).round().astype("Int64")

    l2_summary = l2_summary.drop(columns=["min_id", "max_id"]).sort_values("n_cells", ascending=False)

    if verbose:
        print(l2_summary.to_string(index=False))

    # -----------------------------
    # Step 6: Example Labels (First 5 per L2)
    # -----------------------------
    if verbose:
        print("\n\nExample L3 Labels (First 5 per L2 Cell Type):")
        print("-" * 80)

    example_labels = (
        df[[l2_key, l3_key, subcluster_key]]
          .drop_duplicates()
          .sort_values([l2_key, subcluster_key])
          .groupby(l2_key, as_index=False)
          .head(5)
    )

    if verbose:
        print(example_labels.to_string(index=False))
        print("\n✅ L3 label standardization complete!")
        print(line + "\n")

    return adata, l2_summary, example_labels


# ============================
# Usage (assumes you have adata)
# ============================
adata, l2_summary, example_labels = standardize_myeloid_l3_labels(
    adata,
    l2_key="cell_type_L2",
    l3_key="cell_type_L3",
    verbose=True
)


In [ ]:
# ===== Check Available Columns and QC Metrics =====
print(f"\nKey metadata columns:")
if CELLTYPE_COL in adata.obs.columns:
    print(f"  ✓ Cell type column: {CELLTYPE_COL}")
    celltype_counts = adata.obs[CELLTYPE_COL].value_counts()
    print(f"    Unique cell types: {len(celltype_counts)}")
else:
    print(f"  ❌ Cell type column '{CELLTYPE_COL}' not found!")
    print(f"  Available columns: {list(adata.obs.columns)}")
    raise ValueError(f"Required column '{CELLTYPE_COL}' not found in adata.obs")

if BATCH_KEY in adata.obs.columns:
    print(f"  ✓ Batch key: {BATCH_KEY}")
    print(f"    Unique batches: {adata.obs[BATCH_KEY].nunique()}")
    BATCH_KEY_AVAILABLE = True
else:
    BATCH_KEY_AVAILABLE = False
    print(f"  ⚠️  Batch key '{BATCH_KEY}' not found - skipping batch UMAP")

# Check QC columns
qc_cols = [col for col in adata.obs.columns if any(x in col.lower() for x in ['total_counts', 'n_genes', 'pct_counts_mt'])]
if qc_cols:
    print(f"\n  QC metrics available: {qc_cols}")
else:
    print(f"\n  ⚠️  No standard QC metrics found in .obs")

In [ ]:
# ===== Display Current Annotation Distribution =====
if CELLTYPE_COL in adata.obs.columns:
    print("\nCurrent cell type annotations to validate:")
    print("=" * 80)
    celltype_counts = adata.obs[CELLTYPE_COL].value_counts()
    
    for ct, count in celltype_counts.items():
        pct = count / adata.n_obs * 100
        print(f"  {ct}: {count:,} cells ({pct:.1f}%)")
    
    print("=" * 80)

## 4. Prepare Data for Signature Scoring

In [ ]:
# ===== Ensure log1p layer for scoring =====
print("Preparing data for signature scoring...")

if 'log1p' in adata.layers:
    print("  ✓ Using existing log1p layer")
    score_layer = 'log1p'
elif 'counts' in adata.layers:
    print("  ⚙️  Generating log1p layer from counts")
    # Create temporary normalized layer
    adata.layers['log1p'] = adata.layers['counts'].copy()
    sc.pp.normalize_total(adata, target_sum=1e4, layer='log1p')
    sc.pp.log1p(adata, layer='log1p')
    score_layer = 'log1p'
else:
    print("  ⚠️  No counts/log1p layer found, will use .X")
    score_layer = None

# Set use_raw for scoring
use_raw = adata.raw is not None
gene_names = adata.raw.var_names if use_raw else adata.var_names

# If .raw exists but a scoring layer is available, prefer the layer to avoid conflicts
if use_raw and score_layer is not None:
    print("  ℹ️  .raw present; using log1p layer for scoring (use_raw=False)")
    use_raw = False
    gene_names = adata.var_names

print(f"  Scoring configuration:")
print(f"    use_raw: {use_raw}")
print(f"    layer: {score_layer}")
print(f"    genes available: {len(gene_names)}")

In [ ]:
# ===== Check Marker Availability =====
print("\nChecking marker gene availability...")

def check_marker_availability(marker_dict, gene_names):
    """Check which markers are present in dataset"""
    results = {}
    for category, genes in marker_dict.items():
        present = [g for g in genes if g in gene_names]
        missing = [g for g in genes if g not in gene_names]
        coverage = len(present) / len(genes) * 100 if genes else 0
        results[category] = {
            'present': present,
            'missing': missing,
            'coverage': coverage,
            'n_present': len(present)
        }
    return results

# Check all marker sets
core_check = check_marker_availability(CORE_MYELOID, gene_names)
subtype_check = check_marker_availability(MYELOID_SUBTYPES_ENHANCED, gene_names)
state_check = check_marker_availability(MYELOID_STATES, gene_names)
contam_check = check_marker_availability(CONTAMINATION_MARKERS_ENHANCED, gene_names)

# Summary
print("\nCore Myeloid Markers:")
for cat, info in core_check.items():
    print(f"  {cat}: {info['n_present']}/{len(info['present']) + len(info['missing'])} ({info['coverage']:.0f}%)")

print("\nMyeloid Subtypes (showing low coverage only):")
for cat, info in subtype_check.items():
    if info['coverage'] < 50:
        print(f"  ⚠️  {cat}: {info['n_present']}/{len(info['present']) + len(info['missing'])} ({info['coverage']:.0f}%)")

print("\nFunctional States:")
for cat, info in state_check.items():
    print(f"  {cat}: {info['n_present']}/{len(info['present']) + len(info['missing'])} ({info['coverage']:.0f}%)")

print("\nContamination Markers (showing low coverage only):")
for cat, info in contam_check.items():
    if info['coverage'] < 70:
        print(f"  ⚠️  {cat}: {info['n_present']}/{len(info['present']) + len(info['missing'])} ({info['coverage']:.0f}%)")

## 5. Signature Scoring

**Why signature scoring?**
- More robust than single genes (handles technical noise)
- Better for doublet detection (dual-signature enrichment)
- Enables quantitative confidence assessment
- Allows cluster-level comparisons via heatmaps

In [ ]:
# ===== Compute Signature Scores =====
print("Computing signature scores...")
print("This may take a few minutes for large datasets\n")

def get_present_genes(genes, gene_names):
    """Return genes present in dataset"""
    return [g for g in genes if g in gene_names]

# Combine all signatures
ALL_SIGNATURES = {}

# Add with prefixes for easy identification
for cat, genes in CORE_MYELOID.items():
    ALL_SIGNATURES[f'CORE_{cat}'] = genes

for cat, genes in MYELOID_SUBTYPES_ENHANCED.items():
    ALL_SIGNATURES[f'SUB_{cat}'] = genes

for cat, genes in MYELOID_STATES.items():
    ALL_SIGNATURES[f'STATE_{cat}'] = genes

for cat, genes in CONTAMINATION_MARKERS_ENHANCED.items():
    ALL_SIGNATURES[f'CONTAM_{cat}'] = genes

# Compute scores
computed_signatures = []
skipped_signatures = []

for sig_name, genes in ALL_SIGNATURES.items():
    present = get_present_genes(genes, gene_names)
    
    # Require at least 3 genes for robust scoring
    if len(present) < 3:
        skipped_signatures.append((sig_name, len(present)))
        continue
    
    try:
        score_kwargs = dict(
            adata=adata,
            gene_list=present,
            score_name=f'score_{sig_name}',
            use_raw=use_raw,
        )
        if score_layer is not None and not use_raw:
            score_kwargs['layer'] = score_layer
        sc.tl.score_genes(**score_kwargs)
        computed_signatures.append(sig_name)
    except Exception as e:
        print(f"  ⚠️  Failed to score {sig_name}: {e}")
        skipped_signatures.append((sig_name, len(present)))

print(f"\nScoring complete:")
print(f"  ✓ Computed: {len(computed_signatures)} signatures")
print(f"  ⚠️  Skipped: {len(skipped_signatures)} signatures (< 3 genes or failed)")

if len(skipped_signatures) > 0 and len(skipped_signatures) <= 10:
    print(f"\n  Skipped signatures:")
    for sig, n_genes in skipped_signatures:
        print(f"    {sig}: {n_genes} genes available")

# Get all score columns
score_cols = [col for col in adata.obs.columns if col.startswith('score_')]
print(f"\n  Total score columns in .obs: {len(score_cols)}")

## 6. Cluster-Level Signature Heatmaps

**Interpretation**:
- **High score in expected signature**: Good match (e.g., "Alveolar_Mac" cluster high in SUB_Alveolar_Mac)
- **High score in contamination signature**: Potential doublet/contamination
- **Dual high scores**: Strong doublet evidence (e.g., SUB_Alveolar_Mac + CONTAM_Endothelial both high)

In [ ]:
# ===== Compute Cluster-Level Mean Signatures =====
print("Computing cluster-level mean signatures...")

if len(score_cols) == 0:
    print("❌ No signature scores computed - cannot generate heatmaps")
else:
    if 'score_mean' not in locals():
        # Aggregate by cell type
        score_mean = adata.obs.groupby(CELLTYPE_COL)[score_cols].mean()
    
    # Save to CSV
    score_mean.to_csv(OUTPUT_DIR / 'signature_scores_mean_by_cluster.csv')
    print(f"  ✓ Saved: signature_scores_mean_by_cluster.csv")
    print(f"  Shape: {score_mean.shape[0]} clusters × {score_mean.shape[1]} signatures")

In [ ]:
# ===== Generate Global Signature Heatmap =====
if len(score_cols) > 0:
    print("\nGenerating signature heatmap...")
    
    if 'score_mean' not in locals():
        score_mean = adata.obs.groupby(CELLTYPE_COL)[score_cols].mean()
    
    # Create heatmap
    fig_height = max(10, 0.35 * score_mean.shape[0] + 2)
    fig_width = max(12, 0.25 * score_mean.shape[1] + 4)
    
    plt.figure(figsize=(fig_width, fig_height))
    
    # Clean column names for display
    display_names = [col.replace('score_', '') for col in score_mean.columns]
    score_mean_display = score_mean.copy()
    score_mean_display.columns = display_names
    
    sns.heatmap(
        score_mean_display,
        cmap='RdBu_r',
        center=0,
        cbar_kws={'label': 'Mean Signature Score'},
        xticklabels=True,
        yticklabels=True,
        linewidths=0.5,
        linecolor='lightgray'
    )
    
    plt.title('Signature Scores by Cluster (Mean)', fontsize=14, pad=20)
    plt.xlabel('Signature', fontsize=12)
    plt.ylabel('Cell Type', fontsize=12)
    plt.xticks(rotation=90, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()
    
    plt.savefig(OUTPUT_DIR / 'signature_scores_heatmap_global.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"  ✓ Saved: signature_scores_heatmap_global.png")
    print(f"\n  📊 Look for:")
    print(f"     - Diagonal patterns (cluster matches expected signature)")
    print(f"     - Dual high scores (potential doublets)")
    print(f"     - High contamination scores (cross-lineage expression)")

In [ ]:
# ===== Generate Category-Specific Heatmaps =====
if len(score_cols) > 0:
    print("\nGenerating category-specific heatmaps...")
    
    if 'score_mean' not in locals():
        score_mean = adata.obs.groupby(CELLTYPE_COL)[score_cols].mean()
    
    categories = {
        'CORE': 'Core Myeloid Identity',
        'SUB': 'Myeloid Subtypes',
        'STATE': 'Functional States',
        'CONTAM': 'Contamination Markers'
    }
    
    for prefix, title in categories.items():
        # Select columns for this category
        cat_cols = [col for col in score_cols if col.startswith(f'score_{prefix}_')]
        
        if len(cat_cols) == 0:
            print(f"  ⚠️  No signatures found for {title}")
            continue
        
        # Subset data
        cat_data = score_mean[cat_cols].copy()
        cat_data.columns = [col.replace(f'score_{prefix}_', '') for col in cat_data.columns]
        
        # Create heatmap
        fig_height = max(8, 0.4 * cat_data.shape[0] + 2)
        fig_width = max(10, 0.5 * cat_data.shape[1] + 3)
        
        plt.figure(figsize=(fig_width, fig_height))
        sns.heatmap(
            cat_data,
            cmap='RdBu_r',
            center=0,
            cbar_kws={'label': 'Mean Score'},
            xticklabels=True,
            yticklabels=True,
            linewidths=0.5,
            linecolor='lightgray'
        )
        
        plt.title(f'{title} Signatures by Cluster', fontsize=14, pad=20)
        plt.xlabel('Signature', fontsize=12)
        plt.ylabel('Cell Type', fontsize=12)
        plt.xticks(rotation=45, ha='right', fontsize=10)
        plt.yticks(rotation=0, fontsize=10)
        plt.tight_layout()
        
        filename = f'signature_scores_heatmap_{prefix.lower()}.png'
        plt.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  ✓ Saved: {filename} ({cat_data.shape[1]} signatures)")
    
    print(f"\n  ✓ Category-specific heatmaps complete")

## 7. Annotation Validation Analysis

In [ ]:
# ===== Compute Validation Metrics =====
print("Computing annotation validation metrics...\n")

if CELLTYPE_COL not in adata.obs.columns:
    raise ValueError(f"Required column '{CELLTYPE_COL}' not found in adata.obs")

if len(score_cols) == 0:
    print("❌ No signature scores available - cannot compute validation metrics")
else:
    if 'score_mean' not in locals():
        score_mean = adata.obs.groupby(CELLTYPE_COL)[score_cols].mean()
    
    validation_results = []
    
    for celltype in adata.obs[CELLTYPE_COL].unique():
        mask = adata.obs[CELLTYPE_COL] == celltype
        n_cells = mask.sum()
        
        # Get mean scores for this cluster
        cluster_scores = score_mean.loc[celltype]
        
        # Separate by category
        core_scores = {col: cluster_scores[col] for col in score_cols if col.startswith('score_CORE_')}
        sub_scores = {col: cluster_scores[col] for col in score_cols if col.startswith('score_SUB_')}
        state_scores = {col: cluster_scores[col] for col in score_cols if col.startswith('score_STATE_')}
        contam_scores = {col: cluster_scores[col] for col in score_cols if col.startswith('score_CONTAM_')}
        
        # Compute summary metrics
        mean_core = np.mean(list(core_scores.values())) if core_scores else 0
        max_contam = max(contam_scores.values()) if contam_scores else 0
        max_contam_type = max(contam_scores, key=contam_scores.get).replace('score_CONTAM_', '') if contam_scores else 'None'
        
        # Find best matching subtype
        if sub_scores:
            best_sub = max(sub_scores, key=sub_scores.get)
            best_sub_score = sub_scores[best_sub]
            best_sub_name = best_sub.replace('score_SUB_', '')
        else:
            best_sub_score = 0
            best_sub_name = 'None'
        
        # Validation logic
        if mean_core < 0.3:
            validation_status = 'FAIL'
            confidence = 'LOW'
            reason = 'Low core myeloid signature'
        elif max_contam > 1.0:
            validation_status = 'REVIEW'
            confidence = 'MEDIUM'
            reason = f'High contamination: {max_contam_type} ({max_contam:.2f})'
        elif max_contam > 0.5 and mean_core < 0.7:
            validation_status = 'REVIEW'
            confidence = 'MEDIUM'
            reason = f'Moderate contamination with low core signature'
        else:
            validation_status = 'PASS'
            confidence = 'HIGH'
            reason = 'Clean myeloid signature'
        
        # Store results
        validation_results.append({
            'celltype': celltype,
            'n_cells': n_cells,
            'validation_status': validation_status,
            'confidence': confidence,
            'reason': reason,
            'mean_core_score': mean_core,
            'best_subtype_match': best_sub_name,
            'best_subtype_score': best_sub_score,
            'max_contamination': max_contam,
            'max_contamination_type': max_contam_type,
        })
    
    # Create DataFrame
    validation_df = pd.DataFrame(validation_results)
    validation_df = validation_df.sort_values('confidence', ascending=False)
    
    # Save
    validation_df.to_csv(OUTPUT_DIR / 'annotation_validation_summary.csv', index=False)
    print(f"✓ Saved: annotation_validation_summary.csv\n")
    
    # Display summary
    print("=" * 80)
    print("ANNOTATION VALIDATION SUMMARY")
    print("=" * 80)
    print()
    
    for status in ['PASS', 'REVIEW', 'FAIL']:
        subset = validation_df[validation_df['validation_status'] == status]
        if len(subset) == 0:
            continue
        
        print(f"\n{status} ({len(subset)} clusters):")
        print("-" * 80)
        
        for _, row in subset.iterrows():
            print(f"\n  {row['celltype']} ({row['n_cells']:,} cells)")
            print(f"    Confidence: {row['confidence']}")
            print(f"    Reason: {row['reason']}")
            print(f"    Core score: {row['mean_core_score']:.3f}")
            print(f"    Best match: {row['best_subtype_match']} (score: {row['best_subtype_score']:.3f})")
            if row['max_contamination'] > 0.3:
                print(f"    Max contamination: {row['max_contamination_type']} ({row['max_contamination']:.3f})")
    
    print("\n" + "=" * 80)

## 8. Problem-Specific Validation Panels

In [ ]:
# ===== Analyze Problem-Specific Marker Panels =====
print("\nAnalyzing problem-specific marker panels...\n")

# Map problem types to clusters (you may need to adjust these based on your data)
PROBLEM_CLUSTER_MAPPING = {
    'AM_vs_AM_Endo_doublet': ['Alveolar macrophages_c3'],
    'DC_vs_DC_Epi_doublet': ['DC_c1'],
    'Mono_vs_Mono_B_doublet': ['Non-classical monocytes_c0', 'Non-classical monocytes_c1'],
    'True_Mac_vs_Contaminant': [
        'Intestinal macrophages_c0',
        'Intestinal macrophages_c1',
        'Intestinal macrophages_c2',
        'Intestinal macrophages_c3',
        'Intestinal macrophages_c4'
    ],
}

problem_analysis_results = []

for problem_type, cluster_list in PROBLEM_CLUSTER_MAPPING.items():
    print(f"{'='*80}")
    print(f"Problem Type: {problem_type}")
    print(f"{'='*80}")
    
    panel = PROBLEM_SPECIFIC_PANELS[problem_type]
    
    for cluster_name in cluster_list:
        # Check if cluster exists
        if cluster_name not in adata.obs[CELLTYPE_COL].values:
            print(f"\n  ⚠️  Cluster '{cluster_name}' not found in dataset")
            continue
        
        print(f"\n  Analyzing: {cluster_name}")
        print(f"  {'-'*76}")
        
        mask = adata.obs[CELLTYPE_COL] == cluster_name
        n_cells = mask.sum()
        print(f"    Cells: {n_cells:,}")
        
        # Calculate mean expression for each marker group
        panel_scores = {}
        
        for group_name, genes in panel.items():
            present = get_present_genes(genes, gene_names)
            
            if len(present) == 0:
                print(f"    ⚠️  {group_name}: No markers available")
                panel_scores[group_name] = 0
                continue
            
            # Get expression from log1p layer or raw
            if use_raw:
                expr_data = adata.raw[mask, present].X
            else:
                expr_data = adata[mask, present].X
            
            # Compute mean efficiently
            if sparse.issparse(expr_data):
                mean_score = expr_data.mean()
            else:
                mean_score = np.asarray(expr_data).mean()
            panel_scores[group_name] = mean_score
            
            print(f"    {group_name}: {mean_score:.3f} ({len(present)}/{len(genes)} markers)")
        
        # Interpretation
        if problem_type == 'AM_vs_AM_Endo_doublet':
            am_score = panel_scores.get('AM_side', 0)
            endo_score = panel_scores.get('Endo_side', 0) + panel_scores.get('Lymphatic_Endo', 0)
            
            if am_score > 0.5 and endo_score > 0.5:
                interpretation = "⚠️  LIKELY DOUBLET (both AM and Endo high)"
            elif endo_score > am_score:
                interpretation = "⚠️  LIKELY CONTAMINATION (Endo higher than AM)"
            else:
                interpretation = "✓ Appears to be true AM"
        
        elif problem_type == 'DC_vs_DC_Epi_doublet':
            dc_score = panel_scores.get('DC_core', 0) + panel_scores.get('Migratory_DC', 0)
            epi_score = panel_scores.get('Epi_side', 0)
            
            if dc_score > 0.5 and epi_score > 0.5:
                interpretation = "⚠️  LIKELY DOUBLET (both DC and Epi high)"
            elif epi_score > dc_score:
                interpretation = "⚠️  LIKELY CONTAMINATION (Epi higher than DC)"
            else:
                interpretation = "✓ Appears to be true DC"
        
        elif problem_type == 'Mono_vs_Mono_B_doublet':
            mono_score = panel_scores.get('Mono_side', 0)
            b_score = panel_scores.get('B_Plasma_side', 0)
            
            if mono_score > 0.5 and b_score > 0.5:
                interpretation = "⚠️  LIKELY DOUBLET (both Mono and B high)"
            elif b_score > mono_score:
                interpretation = "⚠️  LIKELY CONTAMINATION (B higher than Mono)"
            else:
                interpretation = "✓ Appears to be true Monocyte"
        
        elif problem_type == 'True_Mac_vs_Contaminant':
            mac_score = panel_scores.get('Core_Mac', 0)
            cross_score = panel_scores.get('Cross_lineage', 0)
            
            if mac_score < 0.5:
                interpretation = "❌ LIKELY NOT MACROPHAGE (low core signature)"
            elif cross_score > 0.5:
                interpretation = "⚠️  LIKELY CONTAMINATION (high cross-lineage)"
            else:
                interpretation = "✓ Appears to be true Macrophage"
        else:
            interpretation = "N/A"
        
        print(f"\n    Interpretation: {interpretation}")
        
        # Store results
        result = {
            'problem_type': problem_type,
            'cluster': cluster_name,
            'n_cells': n_cells,
            'interpretation': interpretation,
        }
        result.update(panel_scores)
        problem_analysis_results.append(result)
    
    print()

# Save results
if problem_analysis_results:
    problem_df = pd.DataFrame(problem_analysis_results)
    problem_df.to_csv(OUTPUT_DIR / 'problem_specific_validation.csv', index=False)
    print(f"✓ Saved: problem_specific_validation.csv")
else:
    print("⚠️  No problem clusters found in dataset")

## 9. UMAP Visualizations

In [ ]:
# ===== UMAP: Current Annotations =====
print("Generating UMAP visualizations...\n")

if 'BATCH_KEY_AVAILABLE' not in locals():
    BATCH_KEY_AVAILABLE = BATCH_KEY in adata.obs.columns

ncols = 2 if BATCH_KEY_AVAILABLE else 1
fig, axes = plt.subplots(1, ncols, figsize=(10*ncols, 8))
if ncols == 1:
    axes = [axes]

sc.pl.umap(
    adata,
    color=CELLTYPE_COL,
    ax=axes[0],
    title='Current Cell Type Annotations',
    legend_loc='right margin',
    legend_fontsize=8,
    show=False
)

if BATCH_KEY_AVAILABLE:
    sc.pl.umap(
        adata,
        color=BATCH_KEY,
        ax=axes[1],
        title='Batch Distribution',
        legend_loc='right margin',
        legend_fontsize=8,
        show=False
    )

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_annotations_and_batch.png', dpi=300, bbox_inches='tight')
plt.show()

print("  ✓ Saved: umap_annotations_and_batch.png")

In [ ]:
# ===== UMAP: Key Signature Scores =====
if len(score_cols) > 0:
    print("\nGenerating signature score UMAPs...")
    
    # Select key signatures to visualize
    key_signatures = [
        'score_CORE_Pan_Myeloid_Basic',
        'score_CORE_APC_Core',
        'score_SUB_Alveolar_Mac',
        'score_SUB_Classical_Mono',
        'score_SUB_cDC2',
        'score_STATE_Type_I_IFN_Response',
        'score_CONTAM_Epithelial_General',
        'score_CONTAM_Endothelial',
    ]
    
    # Filter to available signatures
    available_key_sigs = [sig for sig in key_signatures if sig in adata.obs.columns]
    
    if available_key_sigs:
        n_sigs = len(available_key_sigs)
        ncols = 3
        nrows = (n_sigs + ncols - 1) // ncols
        
        fig, axes = plt.subplots(nrows, ncols, figsize=(8*ncols, 6*nrows))
        axes = axes.flatten() if nrows > 1 else [axes] if nrows == 1 and ncols == 1 else axes
        
        for idx, sig in enumerate(available_key_sigs):
            display_name = sig.replace('score_', '').replace('_', ' ')
            
            sc.pl.umap(
                adata,
                color=sig,
                ax=axes[idx],
                title=display_name,
                vmax='p99',
                show=False
            )
        
        # Hide unused subplots
        for idx in range(n_sigs, len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'umap_key_signature_scores.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  ✓ Saved: umap_key_signature_scores.png ({len(available_key_sigs)} signatures)")
    else:
        print(f"  ⚠️  None of the key signatures were computed")

## 10. QC Metrics Analysis (if available)

In [ ]:
# ===== QC Metrics by Cluster =====
print("Analyzing QC metrics by cluster...\n")

qc_cols = [col for col in adata.obs.columns if any(x in col.lower() for x in ['total_counts', 'n_genes', 'pct_counts_mt'])]

if len(qc_cols) == 0:
    print("  ⚠️  No standard QC metrics found")
    print("  Available columns that might be QC-related:")
    numeric_cols = adata.obs.select_dtypes(include=[np.number]).columns.tolist()
    print(f"    {numeric_cols[:10]}...")
else:
    print(f"  Found QC metrics: {qc_cols}\n")
    
    # Compute mean by cluster
    qc_mean = adata.obs.groupby(CELLTYPE_COL)[qc_cols].mean()
    qc_mean.to_csv(OUTPUT_DIR / 'qc_metrics_by_cluster.csv')
    print(f"  ✓ Saved: qc_metrics_by_cluster.csv")
    
    # Visualize
    fig, axes = plt.subplots(1, min(len(qc_cols), 3), figsize=(8*min(len(qc_cols), 3), 6))
    if len(qc_cols) == 1:
        axes = [axes]
    
    for idx, col in enumerate(qc_cols[:3]):
        sc.pl.umap(
            adata,
            color=col,
            ax=axes[idx] if len(qc_cols) > 1 else axes[0],
            title=col,
            vmax='p99',
            show=False
        )
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'umap_qc_metrics.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ Saved: umap_qc_metrics.png")
    
    # Check for doublet indicators (high UMI + high gene count)
    if any('total_counts' in col.lower() or 'n_counts' in col.lower() for col in qc_cols):
        print(f"\n  💡 Tip: Doublets often show high UMI counts AND high gene counts")
        print(f"         Compare QC metrics with contamination scores")

## 11. Final Summary & Recommendations

In [ ]:
# ===== Generate Final Summary Report =====
print("\n" + "="*80)
print("ANNOTATION VALIDATION COMPLETE - FINAL SUMMARY")
print("="*80)
print()

# Summary statistics
if len(score_cols) > 0 and 'validation_df' in locals():
    n_total = len(validation_df)
    n_pass = (validation_df['validation_status'] == 'PASS').sum()
    n_review = (validation_df['validation_status'] == 'REVIEW').sum()
    n_fail = (validation_df['validation_status'] == 'FAIL').sum()
    
    print(f"Validation Summary:")
    print(f"  Total clusters: {n_total}")
    print(f"  ✓ PASS: {n_pass} ({n_pass/n_total*100:.1f}%)")
    print(f"  ⚠️  REVIEW: {n_review} ({n_review/n_total*100:.1f}%)")
    print(f"  ❌ FAIL: {n_fail} ({n_fail/n_total*100:.1f}%)")
    print()

print("Output Files Generated:")
print("-" * 80)
print("  📊 Signature Analysis:")
print("     - signature_scores_mean_by_cluster.csv")
print("     - signature_scores_heatmap_global.png")
print("     - signature_scores_heatmap_*.png (by category)")
print()
print("  ✅ Validation Reports:")
print("     - annotation_validation_summary.csv")
if problem_analysis_results:
    print("     - problem_specific_validation.csv")
print()
print("  🗺️  Visualizations:")
print("     - umap_annotations_and_batch.png")
if len(score_cols) > 0:
    print("     - umap_key_signature_scores.png")
if len(qc_cols) > 0:
    print("     - umap_qc_metrics.png")
    print("     - qc_metrics_by_cluster.csv")
print("-" * 80)
print()

print("Key Interpretation Guidelines:")
print("-" * 80)
print("  📈 Signature Heatmaps:")
print("     - Look for diagonal patterns (cluster matches expected signature)")
print("     - Dual high scores suggest doublets")
print("     - High CONTAM scores indicate cross-lineage expression")
print()
print("  ✅ Validation Metrics:")
print("     - PASS: Clean myeloid signature, low contamination")
print("     - REVIEW: Moderate evidence of issues, needs manual inspection")
print("     - FAIL: Low core myeloid signature, likely contamination")
print()
print("  🎯 Next Steps:")
print("     1. Review clusters flagged as REVIEW or FAIL")
print("     2. Check problem_specific_validation.csv for targeted clusters")
print("     3. Compare signature scores with QC metrics (if available)")
print("     4. Consider filtering/re-annotating flagged clusters")
print("="*80)

print(f"\n📁 All results saved to: {OUTPUT_DIR}")
print("="*80)